# System Definition of Stock-Driven MISO

## Stock & GAS

|  Variable | Description| Dimensions| Definition | Filename |
| ---- | --------|----- |----------- | ---|
|$SF\_10\_base\_year_{g,c,t}$   | initial stock at time t=-1| $r,g,c,t$  | exogenous input | MISO2_Stock_Initialization_v.xlsx   |
|$SF\_10\_projection_{g,t}$  | function stock projections| $r,g,t$ | exogenous input | MISO2_Function_Stock_v.xlsx |
|$sf_{g,c,t}$ |survival functions for the function stock| $r,g,t,c$ | exogenous input calculated with log normal distribution | MISO2_Lifetimes_v.xlsx ($r,g,t$), <br> MISO2_Lifetimes_deviation_v.xlsx ($r,g,t$)|
|$material\_intensity_{m,g,c}$| material intensity per functional unit| $r,m,g,c$ | exogenous input| MISO2_MaterialIntensity_Functions_v.xlsx|
|$SF\_10_{g,t,c}$ | In use function stock| $r,g,t,c$| uses ODYM compute_stock_driven_model_initialstock(self, InitialStock, SwitchTime, NegativeInflowCorrect=False/True) to compute the inuse stock| |
|$FF\_GAS_{g,t}$ | Gross additions to function stock | $r,g,t$ | Output from ODYM compute_stock_driven_model_initialstock(InitialStock,
  SwitchTime, NegativeInflowCorrect=True/False)||
|$FF\_EoL_{g,t,c}$ | EoL flow of function stock | $r,g,t,c$ | Output from ODYM  compute_stock_driven_model_initialstock(InitialStock,
  SwitchTime, NegativeInflowCorrect=False/True) ||
|$F\_10\_11_{m,g,t}$ | EoL flow of material stock | $r,m,g,t$ | $\sum_c FF\_EoL_{g,t,c}*material\_intensity_{m,g,c}$||
|$S\_10_{m,g,t,c}$ | In use material stock, to be calculated once in final year| $r,m,g,t,c$| $SF\_10_{g,t,c}*material\_intensity_{m,g,c}$ | |
|$F\_9\_10\_total_{m,g,t}$ | Gross additions to material stock | $r,m,g,t$ | $FF\_GAS_{g,t}*material\_intensity_{m,g,c=t}$||


If $FF\_GAS_{g,t}$ < 0 when NegativeInflowCorrect = True, than there is an increase in Stock outflow such that the GAS is 0, so there is premature demolition of stock so that the projection is met. If NegativeInflowCorrect = False, than the GAS can become negative. There should be a third option, currently not implemented in ODYM. There the GAS is set to 0, but there is no premature demolition. Meaning that the current stock level can be higher than the projection. Resulting in unused,"sleeping" stock. There will be no addition to stocks until the stocks decays below the projection. There should be three options: negative_inflow = "allow" ODYM False variant | "demolish" equivalent to ODYM True | "sleep". Default is "sleep". 

A third ("sleeping stock") variant of the compute_stock_model function is needed!
- If the surviving stock exceeds the projection, `FF_GAS = 0`, nothing is demolished early, and the stock stays above the projection until survival brings it below. 
- The sleeping stock keeps ageing and leaves its normal survival curve (and so produce `F_10_11`)
- It does contribute to $S\_10$ and the energy capacity check!

For the inuse stock use somthing similar to 

RECC_dsm = dsm.DynamicStockModel(t=np.arange(0,Nc,1), s=FutureStock.copy(), lt = lt)  # The lt parameter is not used, the sf array is handed over directly in the next step.  

Var_S, Var_O, Var_I, IFlags = RECC_dsm.compute_stock_driven_model_initialstock(InitialStock, SwitchTime, NegativeInflowCorrect=False/True)

The NegtiveInflowCorrect should be toggled by an argument when you run the MISO function. 

## Waste rates, Trade shares, Recycling Rates

|  Variable | Description | Dimensions  | Definition | Filename |
| ---- | ------------- |------|----- | --|
|$uw\_de_{m,t}$| unrecoverable waste rate from extraction| $r,m,t$ | exogenous data| MISO2_WasteRate_unrecov_p1_v.xlsx|
|$rw\_rm_{m,t}$| recoverable waste rate for raw materials| $r,m,t$ | exogenous data| MISO2_WasteRate_recov_p3_v.xlsx|
|$uw\_rm_{m,t}$| unrecoverable waste rate for raw materials| $r,m,t$ | exogenous data| MISO2_WasteRate_unrecov_p3_v.xlsx|
|$rw\_rp_{m,t}$| recoverable waste rate for raw product| $r,m,t$ | exogenous data| MISO2_WasteRate_recov_p5_v.xlsx|
|$uw\_rp_{m,t}$| unrecoverable waste rate for raw products| $r,m,t$ | exogenous data| MISO2_WasteRate_unrecov_p5_v.xlsx|
|$rw\_s_{m,g,t}$| recoverable waste rate for semi-finished product| $r,m,g,t$ | exogenous data| MISO2_WasteRate_recov_p7_v.xlsx|
|$uw\_s_{m,g,t}$| unrecoverable waste rate for semi-finished products| $r,m,g,t$ | exogenous data| MISO2_WasteRate_unrecov_p7_v.xlsx| 
|$rw\_f_{m,g,t}$| recoverable waste rate for final product| $r,m,g,t$ | exogenous data| MISO2_WasteRate_recov_p9_v.xlsx|
|$uw\_f_{m,g,t}$| unrecoverable waste rate for final product| $r,m,g,t$ | exogenous data| MISO2_WasteRate_unrecov_p9_v.xlsx| 
|$IM\_share\_rm_{m,t}$| Import share of raw materials| $r,m,t$ | exogenous data| MISO2_Import_p2_shares_v.xlsx|
|$EX\_share\_rm_{m,t}$| Export share of raw materials| $r,m,t$ | exogenous data| MISO2_Export_p2_shares_v.xlsx|
|$IM\_share\_rp_{m,t}$| Import share of raw products products| $r,m,t$ | exogenous data| MISO2_Import_p4_shares_v.xlsx|
|$EX\_share\_rp_{m,t}$| Export share of raw products| $r,m,t$ | exogenous data| MISO2_Export_p4_shares_v.xlsx|
|$IM\_share\_s_{m,t}$| Import share of semi-finished products| $r,m,t$ | exogenous data| MISO2_Import_p6_shares_v.xlsx|
|$EX\_share\_s_{m,t}$| Export share of semi-finished products| $r,m,t$ | exogenous data| MISO2_Export_p6_shares_v.xlsx|
|$IM\_share\_f_{m,g,t}$| Import share of final products| $r,m,g,t$ | exogenous data| MISO2_Import_p8_shares_v.xlsx|
|$EX\_share\_f_{m,g,t}$| Export share of final products| $r,m,g,t$ | exogenous data| MISO2_Export_p8_shares_v.xlsx|
|$IM\_share\_w_{m,t}$| Import share of waste| $r,m,t$ | exogenous data| MISO2_Import_p12_shares_v.xlsx|
|$EX\_share\_w_{m,t}$| Export share of waste| $r,m,t$ | exogenous data| MISO2_Export_p12_shares_v.xlsx|
|$rate\_collected_{m,g,t}$ | Collection rate of EOL Flows| $r,m,g,t$ | exogenous data| MISO2_RateCollection_v.xlsx |
|$rate\_recycling_{m,g,t}$ | Recycling rate| $r,m,g,t$ | exogenous data| MISO2_RateRecycling_v.xlsx |
|$rate\_downcycling_{m,g,t}$ | Downcycling rate| $r,m,g,t$ | exogenous data| MISO2_RateDowncycling_v.xlsx |
|$rate\_energy\_recovery_{m,g,t}$ | Energy recovery rate| $r,m,g,t$ | exogenous data| MISO2_RateEnergyRecovery_v.xlsx |
|$new\_scrap\_recycling_m$ | Recycling rate for new scrap | $r,m$ | exogenous data | MISO2_stock_driven_properties.xlsx in sheet SB_Material row New_scrap_cycle. |


## Optional Input for trade

Optional input into the model. If it doesn't exist imports are assumed to have the same share of secondary materials as domestic material flows. This option is choosen if the parameters $IM\_share\_sec\_rp_{m,t},IM\_share\_sec\_s_{m,t}$ and $IM\_share\_sec\_f_{m,g,t}$ exist in the MISO config, Meaning that the input files were listed in the MISO2_stock_driven_Config_v.xlsx. 

|  Variable | Description | Dimensions  | Definition | Filename |
| ---- | ------------- |------|----- | --|
|$IM\_share\_sec\_rp_{m,t}$| Share of raw products products that are imported and from secondary materials.| $r,m,t$ | exogenous data| MISO2_Import_p4_shares_secondary_v.xlsx|
|$IM\_share\_sec\_s_{m,t}$| Share of semi-finished products products that are imported and from secondary materials. | $r,m,t$ | exogenous data| MISO2_Import_p6_shares_secondary_v.xlsx|
|$IM\_share\_sec\_f_{m,g,t}$| Share of final products products that are imported and from secondary materials. | $r,m,g,t$ | exogenous data| MISO2_Import_p8_shares_secondary_v.xlsx|

## Total Flows

|  Variable | Description | Dimensions  | Definition |
| ---- | ----------|--- |----------- |
|$F\_9\_10\_total_{m,g,t}$ | Gross addition to material stock.| $r,m,g,t$ | $FF\_GAS_{g,t}*material\_intensity_{m,g,c=t}$|
|$F\_8\_9\_total_{m,g,t}$ | Final products| $r,m,g,t$ | $F\_9\_10\_total_{m,g,t}/(1-(rw\_f_{m,g,t}+uw\_f_{m,g,t}))$|
|$F\_18\_8\_total_{m,g,t}$| Import of final products| $r,m,g,t$ | $F\_8\_9\_total_{m,g,t}*IM\_share\_f_{m,g,t}$|
|$F\_8\_18\_total_{m,g,t}$| Export of final products| $r,m,g,t$ | $F\_8\_9\_total_{m,g,t}*EX\_share\_f_{m,g,t}$|
|$F\_7\_8\_total_{m,g,t}$ | Final products pre market| $r,m,g,t$ | $F\_8\_9\_total_{m,g,t}+F\_8\_9\_total_{m,g,t} * (EX\_share\_f_{m,g,t}-IM\_share\_f_{m,g,t})$|
|$F\_6\_7\_total_{m,g,t}$ | Semi-finished products| $r,m,g,t$ | $F\_7\_8\_total_{m,g,t}/(1-(rw\_s_{m,g,t}+uw\_s_{m,g,t}))$|
|$F\_17\_6\_total_{m,g,t}$| Import of semi-finished products| $r,m,g,t$ | $F\_6\_7\_total_{m,g,t}*IM\_share\_s_{m,t}$|
|$F\_6\_17\_total_{m,g,t}$| Export of semi-finished products| $r,m,g,t$ | $F\_6\_7\_total_{m,g,t}*EX\_share\_s_{m,t}$| 
|$F\_5\_6\_total_{m,g,t}$ | Semi-finished products pre market| $r,m,g,t$ | $F\_6\_7\_total_{m,g,t}+F\_6\_7\_total_{m,g,t} * (EX\_share\_s_{m,t}-IM\_share\_s_{m,t})$|
|$F\_4\_5\_total_{m,g,t}$ | Raw products| $r,m,g,t$ | $F\_5\_6\_total_{m,g,t}/(1-(rw\_rp_{m,t}+uw\_rp_{m,t}))$|
|$F\_16\_4\_total_{m,g,t}$| Import of raw products| $r,m,g,t$ | $F\_4\_5\_total_{m,g,t}*IM\_share\_rp_{m,t}$|
|$F\_4\_16\_total_{m,g,t}$| Export of raw products| $r,m,g,t$ | $F\_4\_5\_total_{m,g,t}*EX\_share\_rp_{m,t}$| 
|$required\_total\_production_{m,g,t}$ | Raw products pre market| $r,m,g,t$ | $F\_4\_5\_total_{m,g,t}+F\_4\_5\_total_{m,g,t} * (EX\_share\_rp_{m,t}-IM\_share\_rp_{m,t})$|
   


## Secondary Flows

|  Variable | Description| Dimensions   | Definition |
| ---- | ----------|--- |----------- |
|$F\_13\_3\_recyc/downcyc_{m,g,t}$ | Flow of recycled/downcylced materials| $r,m,g,t$ ||
|$F\_3\_4\_sec_{m,g,t}=F\_3\_4_{e=1,m,g,t}$ | Secondary material production | $r,m,g,t$ |$(F\_13\_3\_recyc_{m,g,t}+F\_13\_3\_downcyc_{m,g,t})*(1-rw\_rm_{m,t})$| 
|$secondary\_share_{m,g,t}$ | Ratio of supply of secondary materials compared to the total raw product flow| $r,m,g,t$ |$F\_3\_4\_sec_{m,g,t}/required\_total\_production_{m,g,t}$ if $required\_total\_production_{m,g,t}=0,\ secondary\_share_{m,g,t} =0 $|


**If there is no additional input about the secondary share of imports**


|  Variable | Description| Dimensions   | Definition |
| ---- | ----------|--- |----------- |
|$F\_16\_4\_sec_{m,g,t}=F\_16\_4_{e=1,m,g,t}$| Import of raw products from secondary materials| $r,e,m,g,t$ | $F\_16\_4\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_4\_16\_sec_{m,g,t}=F\_4\_16_{e=1,m,g,t}$| Export of raw products from secondary materials| $r,e,m,g,t$ | $F\_4\_16\_total_{m,g,t}* secondary\_share_{m,g,t}$| 
|$F\_4\_5\_sec_{m,g,t}=F\_4\_5_{e=1,m,g,t}$ | Flow of raw products post trade from secondary materials| $r,e,m,g,t$ | $F\_4\_5\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_5\_6\_sec_{m,g,t}=F\_5\_6_{e=1,m,g,t}$ |Flow of semi-finished products from secondary materials, pre market| $r,e,m,g,t$ | $F\_5\_6\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_6\_7\_sec_{m,g,t}=F\_6\_7_{e=1,m,g,t}$ | Flow of semi-finished products from secondary materials post market| $r,e,m,g,t$ | $F\_6\_7\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_17\_6\_sec_{m,g,t}=F\_17\_6_{e=1,m,g,t}$| Import of semi-finished products from secondary materials| $r,e,m,g,t$ | $F\_17\_6\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_6\_17\_sec_{m,g,t}=F\_6\_17_{e=1,m,g,t}$| Export of semi-finished products from secondary materials| $r,e,m,g,t$ | $F\_6\_17\_total_{m,g,t}* secondary\_share_{m,g,t}$|  
|$F\_7\_8\_sec_{m,g,t}=F\_7\_8_{e=1,m,g,t}$ |Flow of final products, pre market| $r,e,m,g,t$ |$F\_7\_8\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_18\_8\_sec_{m,g,t}=F\_18\_8_{e=1,m,g,t}$| Import of final products from secondary materials| $r,e,m,g,t$ | $F\_18\_8\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_8\_18\_sec_{m,g,t}=F\_8\_18_{e=1,m,g,t}$| Export of final products from secondary materials| $r,e,m,g,t$ | $F\_8\_18\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_8\_9\_sec_{m,g,t}=F\_8\_9_{e=1,m,g,t}$ | Final products from secondary materials| $r,e,m,g,t$ | $F\_8\_9\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_9\_10\_sec_{m,g,t}=F\_9\_10_{e=1,m,g,t}$ | Gross addition to material stock from secondary materials| $r,e,m,g,t$ | $F\_9\_10\_total_{m,g,t}* secondary\_share_{m,g,t}$|

**If there is additional input about the secondary share of imports**

If we have additional information about the share of secondary materials in imports we can include this information to adapt the share of secondary materials within the economy. 



|  Variable | Description| Dimensions   | Definition |
| ---- | ----------|--- |----------- |
|$F\_16\_4\_sec_{m,g,t}=F\_16\_4_{e=1,m,g,t}$| Import of raw products from secondary materials| $r,e,m,g,t$ | $F\_16\_4\_total_{m,g,t}*IM\_share\_sec\_rp_{m,t}$|
|$F\_4\_16\_sec_{m,g,t}=F\_4\_16_{e=1,m,g,t}$| Export of raw products from secondary materials| $r,e,m,g,t$ | $F\_4\_16\_total_{m,g,t}* secondary\_share_{m,g,t}$|
|$F\_4\_5\_sec_{m,g,t}=F\_4\_5_{e=1,m,g,t}$ | Flow of raw products post trade from secondary materials| $r,e,m,g,t$ | $(F\_13\_3\_recyc_{m,g,t}+F\_13\_3\_downcyc_{m,g,t})*(1-rw\_rm_{m,t})+F\_16\_4\_sec_{e,m,g,t}-F\_4\_16\_sec_{e,m,g,t}$|
|$F\_5\_6\_sec_{m,g,t}=F\_5\_6_{e=1,m,g,t}$ |Flow of semi-finished products from secondary materials, pre market| $r,e,m,g,t$ | $F\_4\_5\_sec_{m,g,t}-F\_4\_5\_sec_{m,g,t}*(rw\_rp_{m,t}+uw\_rp_{m,t})$|
|$F\_17\_6\_sec_{m,g,t}=F\_17\_6_{e=1,m,g,t}$| Import of semi-finished products from secondary materials| $r,e,m,g,t$ | $F\_17\_6\_total_{m,g,t}*IM\_share\_sec\_s_{m,t}$|
|$F\_6\_17\_sec_{m,g,t}=F\_6\_17_{e=1,m,g,t}$| Export of semi-finished products from secondary materials| $r,e,m,g,t$ | $F\_6\_17\_total_{m,g,t}*(F\_5\_6\_sec_{m,g,t}/F\_5\_6\_total_{m,g,t})$ if $F\_5\_6\_total_{m,g,t}=0,\ (F\_5\_6\_sec_{m,g,t}/F\_5\_6\_total_{m,g,t}) =0 $|  
|$F\_6\_7\_sec_{m,g,t}=F\_6\_7_{e=1,m,g,t}$ | Flow of semi-finished products from secondary materials post market| $r,e,m,g,t$ | $F\_5\_6\_sec_{m,g,t}+F\_17\_6\_sec_{m,g,t}-F\_6\_17\_sec_{m,g,t}$|
|$F\_7\_8\_sec_{m,g,t}=F\_7\_8_{e=1,m,g,t}$ |Flow of final products, pre market| $r,e,m,g,t$ |$F\_6\_7\_sec_{m,g,t}-F\_6\_7\_sec_{m,g,t}*(rw\_s_{m,g,t}+uw\_s_{m,g,t})$|
|$F\_18\_8\_sec_{m,g,t}=F\_18\_8_{e=1,m,g,t}$| Import of final products from secondary materials| $r,e,m,g,t$ | $F\_18\_8\_total_{m,g,t}*IM\_share\_sec\_f_{m,g,t}$|
|$F\_8\_18\_sec_{m,g,t}=F\_8\_18_{e=1,m,g,t}$| Export of final products from secondary materials| $r,e,m,g,t$ | $F\_8\_18\_total_{m,g,t}*(F\_7\_8\_sec_{m,g,t}/F\_7\_8\_total_{m,g,t})$ if $F\_7\_8\_total_{m,g,t}=0,\ (F\_7\_8\_sec_{m,g,t}/F\_7\_8\_total_{m,g,t}) =0 $|
|$F\_8\_9\_sec_{m,g,t}=F\_8\_9_{e=1,m,g,t}$ | Final products from secondary materials| $r,e,m,g,t$ | $F\_7\_8\_sec_{m,g,t}+F\_18\_8\_sec_{m,g,t}-F\_8\_18\_sec_{m,g,t}$|
|$F\_9\_10\_sec_{m,g,t}=F\_9\_10_{e=1,m,g,t}$ | Gross addition to material stock from secondary materials| $r,e,m,g,t$ | $F\_8\_9\_sec_{m,g,t}-F\_8\_9\_sec_{m,g,t}*(rw\_f_{m,g,t}+uw\_f_{m,g,t})$|

Regarding secondary flow above total this will be taken care in the import files. Check post market if the secondary flow is above the total flow or below zero. If it is above the total flow there should be a warning. If the secondary flow drops below zero clamp it to 0 and warn. The Export flows should absorb the difference. 

## Primary Flows

|  Variable | Description| Dimensions   | Definition |
| ---- | ----------|--- |----------- |
|$F\_3\_4\_prim_{m,g,t}=F\_3\_4_{e=0,m,g,t}$ | Raw material production | $r,m,g,t$ |$required\_total\_production_{m,g,t}-F\_3\_4\_sec_{m,g,t}$| 
|$F\_2\_3\_prim_{m,g,t}$ | Flow of raw materials| $r,m,g,t$ | $F\_3\_4\_prim_{m,g,t}/(1-(rw\_rm_{m,t}+uw\_rm_{m,t}))$|
|$F\_15\_2\_prim_{m,g,t}$| Import of raw materials| $r,m,g,t$ | $F\_2\_3\_prim_{m,g,t}*IM\_share\_rm_{m,t}$|
|$F\_2\_15\_prim_{m,g,t}$| Export raw materials| $r,m,g,t$ | $F\_2\_3\_prim_{m,g,t}*EX\_share\_rm_{m,t}$|
|$F\_1\_2\_prim_{m,g,t}$ | Raw materials pre market| $r,m,g,t$ | $F\_2\_3\_prim_{m,g,t}+F\_2\_3\_prim_{m,g,t} * (EX\_share\_rm_{m,t}-IM\_share\_rm_{m,t})$|   
|$F\_0\_1\_prim_{m,g,t}$ | Domestic Extraction incl. extraction waste| $r,m,g,t$ | $F\_1\_2\_prim_{m,g,t}/(1-uw\_de_{m,t})$|

**For all other flows applies:**

Primary flows $F\_x\_y\_prim_{m,g,t}=F\_x\_y_{e=0,m,g,t}$ =$F\_x\_y\_total_{m,g,t}-F\_x\_y\_sec_{m,g,t}$

## End of Life Flows

|  Variable | Description | Dimensions | Definition |
| ---- | -----------|-- |----------- |   
|$F\_10\_11_{m,g,t}$ | End of Life (EoL) outflows from material stock| $r,m,g,t$ | $ \sum_c FF\_EoL_{g,t,c}*material\_intensity_{m,g,c}$| 
|$stock\_dissipation_{m,g,t}$ | Unrecoverable waste from material stock, dissipation | $r,m,g,t$ | $F\_10\_11_{m,g,t}*(1-rate\_collected_{m,g,t})$| 
|$F\_9\_11\_a_{m,g,t}$ | Recoverable waste during construction| $r,m,g,t$ | $F\_8\_9\_total_{m,g,t}*rw\_f_{m,g,t}$|
|$F\_9\_11\_b_{m,g,t}$ | Unrecoverable waste during construction| $r,m,g,t$ |$F\_8\_9\_total_{m,g,t}*uw\_f_{m,g,t}$|
|$F\_7\_11\_a_{m,g,t}$ | Recoverable waste final product manufacturing| $r,m,g,t$ |$F\_6\_7\_total_{m,g,t}*rw\_s_{m,g,t}$|
|$F\_7\_11\_b_{m,g,t}$ | Unrecoverable waste final product manufacturing| $r,m,g,t$ |$F\_6\_7\_total_{m,g,t}*uw\_s_{m,g,t}$|
|$F\_5\_11\_a_{m,g,t}$ | Recoverable waste semi-finished product fabrication| $r,m,g,t$ |$F\_4\_5\_total_{m,g,t}*rw\_rp_{m,t}$|
|$F\_5\_11\_b_{m,g,t}$ | Unrecoverable waste semi-finished product fabrication| $r,m,g,t$ |$F\_4\_5\_total_{m,g,t}*uw\_rp_{m,t}$|
|$F\_3\_11\_a_{m,g,t}$ | Recoverable waste raw product fabrication| $r,m,g,t$ |$(F\_2\_3\_prim_{m,g,t}+F\_13\_3\_recyc_{m,g,t}+F\_13\_3\_downcyc_{m,g,t})*rw\_rm_{m,t}$|
|$F\_3\_11\_b_{m,g,t}$ | Unrecoverable waste raw product fabrication| $r,m,g,t$ |$F\_2\_3\_prim_{m,g,t}*uw\_rm_{m,t}$|
|$F\_1\_0\_prim_{m,g,t}$ | Extraction waste| $r,m,g,t$ | $F\_1\_2\_prim_{m,g,t} *uw\_de_{m,t}/(1-uw\_de_{m,t})$|   
|$F\_11\_12\_o_{m,g,t}$ | Domestic recoverable old scrap, collected waste from EOL| $r,m,g,t$|$rate\_collected_{m,g,t}*F\_10\_11_{m,g,t}$ |
|$F\_11\_12\_n_{m,g,t}$ | Domestic recoverable new scrap, from industrial processes| $r,m,g,t$|$ F\_9\_11\_a_{m,g,t}+F\_7\_11\_a_{m,g,t}+F\_5\_11\_a_{m,g,t}+F\_3\_11\_a_{m,g,t}$ |
|$F\_11\_14_{m,g,t}$ | Total domestic unrecoverable waste| $r,m,g,t$ |$F\_9\_11\_b_{m,g,t}+F\_7\_11\_b_{m,g,t}+F\_5\_11\_b_{m,g,t}+F\_3\_11\_b_{m,g,t}+F\_20\_11_{m,g,t}+stock\_dissipation_{m,g,t}$ |
|$F\_19\_12\_o_{m,g,t}$| Import of old scrap| $r,m,g,t$ | $F\_11\_12\_o_{m,g,t}*IM\_share\_w_{m,t}$|
|$F\_19\_12\_n_{m,g,t}$| Import of new scrap| $r,m,g,t$ | $F\_11\_12\_n_{m,g,t}*IM\_share\_w_{m,t}$|
|$F\_12\_19\_o_{m,g,t}$| Export of old scrap| $r,m,g,t$ | $F\_11\_12\_o_{m,g,t}*EX\_share\_w_{m,t}$|  
|$F\_12\_19\_n_{m,g,t}$| Export of new scrap| $r,m,g,t$ | $F\_11\_12\_n_{m,g,t}*EX\_share\_w_{m,t}$|  
|$F\_12\_13\_o_{m,g,t}$ | Domestic old scrap supply| $r,m,g,t$ | $F\_11\_12\_o_{m,g,t}+F\_19\_12\_o_{m,g,t}-F\_12\_19\_o_{m,g,t}$|
|$F\_12\_13\_n_{m,g,t}$ | Domestic new scrap supply| $r,m,g,t$ | $F\_11\_12\_n_{m,g,t}+F\_19\_12\_n_{m,g,t}-F\_12\_19\_n_{m,g,t}$|
|$S\_13_{m,g,t=0,c}$|initial waste stock|$r,m,g,t,c$ |= 0|
|$S\_13_{m,g,t,c=t}$ | Stock of recoverable waste | $r,m,g,t,c$ |$F\_12\_13\_o_{m,g,t}+F\_12\_13\_n_{m,g,t}$ |
|$recycled\_new\_scrap_{m,g,t}$ | Secondary materials from new scrap | $r,m,g (target),t$ |$\sum_{g (origin)} F\_12\_13\_n_{m,g,t-1}*new\_scrap\_recycling_m$|
|$F\_13\_3\_recyc_{m,g,t}$ | Flow of recycled raw products| $r,m,g (target),t$ | $\sum_{c=t-max\_age}^{t-1} S\_13_{m,g (origin),t,c}*rate\_recycling_{m,g,t}+recycled\_new\_scrap$, This is in principle the logic applied. There is a more complicated allocation mechanism described below. |
|$F\_13\_3\_downcyc_{m,g,t}$ | Flow of Downcycled raw products| $r,m,g,t$ | $ \sum_{m*} \sum_{c=t-max\_age}^{t-1} S\_13_{m*,g,t,c}*rate\_downcycling_{m*,g,t}$ all materials that have a downcyling rate are pooled to meet demand of aggregates. See mechanism below|
|$F\_13\_20_{m,g,t}$ | Flow of materials for energy recovery| $r,m,g,t$ | $\sum_{c=t-max\_age}^{t-1} S\_13_{m,g,t,c}*rate\_energy\_recovery_{m,g,t}$|
|$maximum\_recycling\_potential_{m,g}$|Share of technically feasible maximum secondary material input per end-use| $r,m,g$ |exogenous data, MISO2_stock_driven_properties.xlsx, Max_InputRecycling  |
|$technical\_recycling\_potential_{m,g,t}$|Technically feasible re/downcyclable potential| $r,m,g,t$ |$required\_total\_production_{m,g,t}*maximum\_recycling\_potential_{m,g}$ |
|$S\_13_{m,g,t,c}$ | Stock of recoverable waste | $r,m,g,t,c$ |$S\_13_{m,g,t−1,c}+F\_12\_13\_o_{m,g,t}+F\_12\_13\_n_{m,g,t}+new\_scrap\_excess\_stock_{m,g,t}-$$(F\_12\_13\_n_{m,g,t}*new\_scrap\_recycling_m + origin\_recycling\_withdrawal_{m,g,t} +\sum_{g'} \sum_{m'}( downcycled\_flow_{m,g,m',g',t})+F\_13\_20_{m,g,t})-F\_13\_14_{m,g,t}$, details see below.|
|$F\_13\_14_{m,g,t}$ | Flow of unused, recoverable waste| $r,m,g,t$ | $S\_13_{m,g,t,c=(t-max\_age)}$ older than MaximumWasteStockAge $=max\_age$|
|$F\_14\_0_{m,g,t}$ | Flow of waste to environment| $r,m,g,t$ | $F\_11\_14_{m,g,t}+F\_13\_14_{m,g,t}$|
|$F\_20\_11_{m,g,t}$ | Flow of waste to environment from energetic processes, emissions and residuals| $r,m,g,t$ | $F\_13\_20_{m,g,t}$|



If $F\_13\_3\_recyc_{m,g,t}+F\_13\_3\_downcyc_{m,g,t}$ > $technical\_recycling\_potential_{m,g,t}$ than $F\_13\_3\_recyc_{m,g,t}+F\_13\_3\_downcyc_{m,g,t} = technical\_recycling\_potential_{m,g,t}$ 

$S\_13_{m,g,t,c}$ is calculated as follows: 
- $S\_13_{m,g,t,c=t-1}=S\_13_{m,g,t,c=t-1}-F\_12\_13\_n_{m,g,t-1}*(new\_scrap\_recycling_m)$, old scrap and unrecycled new scrap is kept in the cohort of the waste stock
- Calculate $recycled\_new\_scrap$, check if it is bigger than $technical\_recycling\_potential_{m,g,t}$ for all materials and end-uses. If it is bigger. Add excess to the waste stock and skip the rest of the recycling steps. If there are some materials where the supply exceeds the recycling demand add these excesses to the waste stock in $c=t-1$ and continue. 
- Calculate $recyclable\_waste=\sum_{c=t-max\_age}^{t-1}S\_13_{m,g,t,c}$
- Calculate $F\_13\_3\_recyc_{m,g,t}$, $F\_13\_3\_downcyc_{m,g,t}$ and $F\_13\_20_{m,g,t}$ 
- Add $(origin\_recycling\_withdrawal_{m,g,t} +\sum_{g'} \sum_{m'}( downcycled\_flow_{m,g,m',g',t})+F\_13\_20_{m,g,t})$ and remove the same amount from the stock, starting in $S\_13_{m,g,t,c=t-max\_age}$ and using the following years until $S\_13_{m,g,t,c=t-1}$ if necessary.
- Waste older than MaximumWasteStockAge is removed from the stock $F\_13\_14_{m,g,t}=S\_13_{m,g,t,c=t-max\_age}$


The enduse information in dimension g will be kept over the whole EOL material chain, in general all recycled materials from all enduses can be used for all enduses as secondary supply. The exceptions are given in MISO2_stock_driven_properties.xlsx in sheet SecondarySupplyException. The columns denote the target end_use while and the rows the origin end_use, all materials named in a cell are the ones that are not allowed to be used in the target end_use if they originate from the row end_use. 

$F\_13\_3\_recyc_{m,g,t}$ is calculated first. 

First calculate possible new scrap quota: Here the SecondarySupplyException rules don't apply. $q\_new\_scrap_{m,t}=min(1,\frac{\sum_g F\_12\_13\_n_{m,g,t-1}*new\_scrap\_recycling_m}{\sum_{g'}technical\_recycling\_potential_{m,g',t}})$ than $recycled\_new\_scrap_{m,g',t}$ = $q\_new\_scrap_{m,t}*technical\_recycling\_potential_{m,g',t}$. 

If $recycled\_new\_scrap_{m,g',t}-technical\_recycling\_potential_{m,g',t}=0$, then old-scrap recycling is skipped for material $m$ and year $t$. The excess is added to the waste stock, $S\_13_{m,g,t,c=t-1}$, as follows:
$recycled\_new\_scrap\_excess_{m,t}=\sum_g F\_12\_13\_n_{m,g,t-1}*new\_scrap\_recycling_m-\sum_{g'} recycled\_new\_scrap_{m,g',t}$
$new\_scrap\_excess\_stock_{m,g,t}=\frac{recycled\_new\_scrap\_excess_{m,t}*F\_12\_13\_n_{m,g,t-1}}{\sum_g(F\_12\_13\_n_{m,g,t-1})}$


$recycled\_flow_{m,g,g',t}$ is calculated in two stages as follows:

- $g$ denotes the origin end use and $g'$ denotes the target end use.

- Calculate the recycling capacity of each origin end use:

  $origin\_recycling\_capacity_{m,g,t}=rate\_recycling_{m,g,t}\cdot recyclable\_waste_{m,g,t}$

- Calculate the remaining recycling demand of each target end use after new-scrap recycling:

  $remaining\_recycling\_demand_{m,g',t}=technical\_recycling\_potential_{m,g',t}-recycled\_new\_scrap_{m,g',t}$

- Calculate the recycling quota:

  $q_{m,t}=\min\left(1,\frac{\sum_g origin\_recycling\_capacity_{m,g,t}}{\sum_{g'}remaining\_recycling\_demand_{m,g',t}}\right)$

  If $\sum_{g'}remaining\_recycling\_demand_{m,g',t}=0$, set $recycled\_flow_{m,g,g',t}=0$ for all origin and target end uses.

- First calculate the old-scrap recycling amount received by each target end use:

  $x^{*}_{m,g',t}=\sum_g recycled\_flow_{m,g,g',t}$

  The following constraints apply:

  $\sum_g recycled\_flow_{m,g,g',t}\leq q_{m,t}\cdot remaining\_recycling\_demand_{m,g',t}$

  $\sum_{g'}recycled\_flow_{m,g,g',t}\leq origin\_recycling\_capacity_{m,g,t}$

  $recycled\_flow_{m,g,g',t}=0$ for origin-target combinations excluded by `SecondarySupplyException`.

  $recycled\_flow_{m,g,g',t}\geq 0$ for allowed origin-target combinations.

- Apply a lexicographic max-min allocation rule to the target end uses:

  1. Raise all target fill ratios together:

     $target\_fill\_ratio_{m,g',t}=\frac{x^{*}_{m,g',t}}{q_{m,t}\cdot remaining\_recycling\_demand_{m,g',t}}$

  2. Freeze a target end use if it cannot receive more material because all allowed origin end uses have no remaining recycling capacity.

  3. Continue raising the fill ratios of the remaining target end uses.

  4. Do not exceed $target\_fill\_ratio_{m,g',t}=1$.

  5. Do not exceed $origin\_recycling\_capacity_{m,g,t}$.

- Keep the resulting target amounts $x^{*}_{m,g',t}$ fixed. Then determine which origin end uses supply these amounts.

- Calculate the recycling withdrawal from each origin end use:

  $origin\_recycling\_withdrawal_{m,g,t}=\sum_{g'}recycled\_flow_{m,g,g',t}$

- Calculate the origin utilisation ratio:

  $origin\_utilization\_ratio_{m,g,t}=\frac{origin\_recycling\_withdrawal_{m,g,t}}{origin\_recycling\_capacity_{m,g,t}}$, only for enduses with non-zero $origin\_recycling\_capacity_{m,g,t}$ otherwise 0. 

- Apply a second lexicographic max-min allocation rule:

  1. Keep $\sum_g recycled\_flow_{m,g,g',t}=x^{*}_{m,g',t}$ fixed for every target end use $g'$.

  2. Raise all $origin\_utilization\_ratio_{m,g,t}$ values together.

  3. Allocate material only through origin-target combinations allowed by `SecondarySupplyException`.

  4. Freeze an origin end use if it cannot supply more material because all reachable target amounts are already fulfilled.

  5. Continue raising the utilisation ratios of the remaining origin end uses.

  6. Stop when all target amounts $x^{*}_{m,g',t}$ are supplied



$F\_13\_3\_recyc_{m,g',t}=recycled\_new\_scrap_{m,g',t}+\sum_g recycled\_flow_{m,g,g',t}$

$F\_13\_3\_downcyc_{m,g',t}$ is calculated as follows: Here there are no constraints between end-uses. Calculate the downcycling demand by $technical\_recycling\_potential_{m,g',t}-F\_13\_3\_recyc_{m,g',t}=downcycling\_demand_{m,g',t}$, if $downcycling\_demand_{m,g',t}$ is 0 skip the rest of the downcycling calculations,  for all materials marked with TRUE as aggregates in the MISO2_stock_driven_properties.xlsx, SB_Material sheet.
$downcycling\_supply\_potential_{m,g,t}=\sum_{c=t-max\_age}^{t-1}(S\_13_{m,g,t,c}*rate\_downcycling_{m,g,t})$
$total\_downcycling\_supply_{t}=\sum_m(\sum_g(downcycling\_supply\_potential_{m,g,t}))$ and $total\_downcycling\_demand_{t}=\sum_{m'}(\sum_{g'}(downcycling\_demand_{m',g',t}))$

The supply is summed over all materials (`Σ_m`), so every material with a downcycling rate competes for the same aggregate demand.- Intended? YES!

Then the fraction that can be provided is $q\_down=min(1,\frac{total\_downcycling\_supply_{t}}{total\_downcycling\_demand_{t}})$, or 0 if $total\_downcycling\_supply_{t}$ or $total\_downcycling\_demand_{t}$ are 0. $F\_13\_3\_downcyc_{m',g',t}=q\_down*downcycling\_demand_{m',g',t}$ and $downcycled\_flow_{m,g,m',g',t}=\frac{downcycling\_supply\_potential_{m,g,t}}{total\_downcycling\_supply_t}*F\_13\_3\_downcyc_{m',g',t}$
All source end uses should contribute equally, pro rata, to the downcycled material supply. The overall demand of all types of aggregates should be aggregated and used to calculate the fraction here. All aggregate types should receive the same share of downcycled aggregate supply. If the supply exceeds the demand the excess should stay in the stock. 

Energy recovery, $F\_13\_20_{m,g,t}$, is calculated by $\sum_{c=t-max\_age}^{t-1} S\_13_{m,g,t,c}*rate\_energy\_recovery_{m,g,t}$. For energy recovery $g=g'$


Intent of the rules:
- Is leaving the unreachable remainder in `S_13` acceptable? YES
- New scrap "the rest of the calculation is skipped" applies only if targets are hit for all `g`


Include warning: $rate\_recycling_{m,g,t}+rate\_downcycling_{m,g,t}+rate\_energy\_recovery_{m,g,t}$>1 Input data needs to be changed. Otherwise mass balance is not conserved. This allows to calculate downcycling and energy recovery after the recycling on the same waste stock. 


If a material is downcyled it will change its properties and be reclassified as aggregates (any of the materials classifed as Aggregates in SB_Material) 

Some materials also change their classification when they are recycled. These materials can be found in MISO2_stock_driven_properties.xlsx, AggrRecyclingSplit. Here the first column gives the material that change their classification and the column header gives the material that they change into. The number in the cell gives the fraction of mass that should be reclassified as the respective material. 
The split should happen when $recyclable\_waste_{m,g,t}$ is calculated, the target recycling rates apply. To calculate the withdrawal from $S\_13$ $origin\_utilization\_ratio_{m,g,t}$
is transferred back so that the Stock is kept in the initial materials as much as possible. If different shares of the target materials were recycled e.g. → 95 t aggregates + 5 t cement. Recycling uses 50% of the aggregates (47.5 t) and 100% of the cement (5 t). Then as much concrete leaves the stock as needed to recycle the cement, and the aggregates are transferred back to the waste stock, keeping the mass balance. If there Recycling uses 50% of the aggregates (47.5 t) and 80% of the cement (4 t), then 20 t concrete and (47.5 -19 (aggregate share of the 20t concrete)t) aggregates are in the wastestock. Or the other way around. The returned material has the cohort of the original material. Returns are done after downcycling and energy recovery for next year.

Include a check if one row doesn't sum to 1 and raise a warning. If that happens mass balance is not preserved. 

The maximum waste stock age is set in MISO2_stock_driven_properties.xlsx, GlobalParameters under MaximumWasteStockAge $=max\_age$.

- The waste trade reference should stay in reference to the current flow. The check should happen on the input data side (if Export share exceeds 1 it should be set to 1).
- Secondary inputs in P3 only lose recoverable waste. 

## Energy + Feedback

$j$ is the index for energy carrier type

Optional, only executed if the input files are added to the MISO_files sheet in the Config excel otherwise omit and start with the next year

|  Variable | Description | Dimensions   | Definition | Filename|
| ---- | -----------|---- |---- |--|
|$operational\_energy\_intensity_{j,g,t}$ | Operational Energy per projected function stock| $r,g,j,t$ | exogenous data| MISO2_Energy_Operational_v.xlsx|
|$operational\_energy_{j,g,t}$ | Operational Energy of projected function stock| $r,g,j,t$ | $operational\_energy\_intensity_{j,g,t}*\sum_c SF\_10_{g,t,c}$| |
|$energy\_recovery\_intensity_{m,t}$ | Energy per mass of material for energy recovery| $r,m,t$ | exogenous data| MISO2_Energy_Intensity_sec_v.xlsx|
|$E\_energy\_recovery_{m,g,j=energy\_recovery,t}$ | Energy from energy recovery| $r,m,g,j=energy\_recovery,t$ | $F\_13\_20_{m,g,t}*energy\_recovery\_intensity_{m,t}$ |
|$energy\_intensity\_p1_{m,j,t}$ | Energy intensity of extraction| $r,m,j,t$ | exogenous data| MISO2_Energy_p1_v.xlsx|
|$E\_20\_1_{m,g,j,t}$ | Energy for domestic extraction| $r,m,g,j,t$ | $F\_1\_2\_prim_{m,g,t}*energy\_intensity\_p1_{m,j,t}$ |
|$energy\_intensity\_p3_{m,j,t}$ | Energy intensity of refining| $r,m,j,t$ | exogenous data| MISO2_Energy_p3_v.xlsx|
|$E\_20\_3_{m,g,j,t}$ | Energy for refining| $r,m,g,j,t$ | $F\_3\_4\_prim_{m,g,t}*energy\_intensity\_p3_{m,j,t}$ |
|$energy\_intensity\_p5_{m,j,t}$ | Energy intensity of fabrication of semi-finished products| $r,m,j,t$ | exogenous data| MISO2_Energy_p5_v.xlsx|
|$E\_20\_5_{m,g,j,t}$ | Energy for fabrication of semi-finished products| $r,m,g,j,t$ | $F\_5\_6\_total_{m,g,t}*energy\_intensity\_p5_{m,j,t}$ |
|$energy\_intensity\_p7_{g,j,t}$ | Energy intensity of manufacturing of final products| $r,g,j,t$ | exogenous data| MISO2_Energy_p7_v.xlsx|
|$E\_20\_7_{m,g,j,t}$ | Energy for manufacturing of final products| $r,m, g,j,t$ | $F\_7\_8\_total_{m,g,t}*energy\_intensity\_p7_{g,j,t}$ |
|$energy\_intensity\_p9_{g,j,t}$ | Energy intensity of construction and assembly| $r,g,j,t$ | exogenous data| MISO2_Energy_p9_v.xlsx|
|$E\_20\_9_{m,g,j,t}$ | Energy for construction and assembly| $r,m,g,j,t$ | $F\_9\_10\_total_{m,g,t}*energy\_intensity\_p9_{g,j,t}$ |
|$energy\_intensity\_p13\_a_{m,j,t}$ | Energy intensity of recycling| $r,m,j,t$ | exogenous data| MISO2_Energy_p13_a_v.xlsx|
|$E\_20\_13\_recyc_{m,g,j,t}$ | Energy for recycling| $r,m,g,j,t$ | $F\_13\_3\_recyc_{m,g,t}*energy\_intensity\_p13\_a_{m,j,t}$ |
|$energy\_intensity\_p13\_b_{m,j,t}$ | Energy intensity of downcycling| $r,m,j,t$ | exogenous data| MISO2_Energy_p13_b_v.xlsx|
|$E\_20\_13\_downcyc_{m,g,j,t}$ | Energy for downcycling| $r,m,g,j,t$ | $F\_13\_3\_downcyc_{m,g,t}*energy\_intensity\_p13\_b_{m,j,t}$ |
|$energy\_intensity\_p13\_c_{m,j,t}$ | Energy intensity of energy recovery| $r,m,j,t$ | exogenous data| MISO2_Energy_p13_c_v.xlsx|
|$E\_20\_13\_energyrecov_{m,g,j,t}$ | Energy for energy recovery| $r,m,g,j,t$ | $F\_13\_20_{m,g,t}*energy\_intensity\_p13\_c_{m,j,t}$ |
|$energy\_intensity\_p11_{m,j,t}$ | Energy intensity of waste collection| $r,m,j,t$ | exogenous data| MISO2_Energy_p11_v.xlsx|
|$E\_20\_11_{m,g,j,t}$ | Energy for waste collection| $r,m,g,j,t$ | $(F\_11\_12\_n_{m,g,t}+F\_11\_12\_o_{m,g,t})*energy\_intensity\_p11_{m,j,t}$ |
|$energy\_intensity\_p14_{m,j,t}$ | Energy intensity of land filling| $r,m,j,t$ | exogenous data| MISO2_Energy_p14_v.xlsx|
|$E\_20\_14_{m,g,j,t}$ | Energy for land filling| $r,m,g,j,t$ | $(F\_13\_14_{m,g,t}+F\_11\_14_{m,g,t})*energy\_intensity\_p14_{m,j,t}$ |
|$E\_material\_processing_{m,g,j,t}$ | Total energy for material processing |$r,m,g,j,t$ | $E\_20\_1_{m,g,j,t}+E\_20\_3_{m,g,j,t}+E\_20\_5_{m,g,j,t}+E\_20\_7_{m,g,j,t}+E\_20\_9_{m,g,j,t}+$ <br> $E\_20\_13\_recyc_{m,g,j,t}+E\_20\_13\_downcyc_{m,g,j,t}+E\_20\_13\_energyrecov_{m,g,j,t}+E\_20\_11_{m,g,j,t}+E\_20\_14_{m,g,j,t}$|
|$E\_total_{g,j,t}$ | Total energy for material processing and operational energy |$r,g,j,t$ | $operational\_energy_{j,g,t}+\sum_m E\_material\_processing_{m,g,j,t}$|
|$E\_demand_{j,t}$ | Energy demand |$r,j,t$|$\sum_{g} E\_total_{g,j,t}-\sum_{g,m} E\_energy\_recovery_{m,g,j=energy\_recovery,t}$|
|$stock\_intensity\_capacity_{g,j,t}$ | Function stock intensity of energy capacity |$r,g,j,t$ | exogenous data| MISO2_Stock_Intensity_Capacity_v.xlsx|
|$stock\_capacity_{g,t}$ | Needed stock for energy capacity| $r,g,t$ | $\sum_j E\_demand_{j,t}*stock\_intensity\_capacity_{g,j,t}$, for $g$ corresponding to $j$ in energy map|
|$delta\_capacity_{g,t}$ | Missing stock for energy capacity | $r,g,t$ | $stock\_capacity_{g,t}-\sum_c SF\_10_{g=energy\_capital,t,c}$, Here the demand for energy carrier j gets compared to the function stock of the corresponding end_use. See Energy_map, more infos below|
|$additional\_capacity_{g=energy,t}$|If $delta\_capacity_{g,t}$ > 0 then, $SF\_10_{g=energy\_capital,t+1}=SF\_10_{g=energy\_capital,t+1}+delta\_capacity_{g,t}$ | $g=energy\_capital,t$ |$SF\_10_{g=energy\_capital,t+1}=SF\_10_{g=energy\_capital,t+1}+delta\_capacity_{g,t}$|

In the file MISO2_stock_driven_properties there is a sheet called Energy_map. In this sheet there is a mapping which Energy_carrier j, is produced by which end_use g. If the capacity is not sufficient, $additional\_capacity_{g,t} > 0$, for the energy_carrier j the corresponding end_use g is increased. If there are several carriers mapped to one end use their $delta\_capacity$ is summed.

This energy feedback should be optional in the model run and be to toggeled with a parameter EnergyFeedback in the Model run function. There should be two different model runners depending on this parameter. That share functions where it's possible and differ wheres necessary. 

In MISO2_stock_driven_properties in the Energy_carrier sheet there is a column EnergyRecovery. Here one carrier is indicated with TRUE. This is the energy type thats indicated here with $j=energy\_recovery$. Here some of the energy demand for this carrier can be offset by the energy flow from energy recovery. 

# Material Split

In P9/construction a few materials a transformed. In the stock and $F\_9\_10$ asphalt and concrete are present. But in P9 asphalt get devided into bitumen and aggr_4asph and concrete into cement and aggr_4concr. The required factors, how much (in percent) of concrete/asphalt is cement/bitumen is an exogenous input from MISO2_stock_driven_properties.xlsx, SplitMaterial. This split happens before for the total flow and after, for the secondary and primary flow, all other calculations in P9 are made. The only flow of the P1–P9 chain with the transformed materials is $F\_9\_10$.
-`F_9_10_{e,concrete} = Σ_{m' ∈ parts(concrete)} F_9_10_{e,m'}`.
- Waste rates of P9 apply to the split materials (cement, `aggr_4concr`), not to concrete.

In P3 the same thing happens for steel. The Splits can be found in MISO2_stock_driven_properties.xlsx, SplitMaterial and in the Sheet SB_Material there are two columns that check if the material should be split and in which process. If there is a split than the total mass of material before the split is reclassified into new materials (it is also possible that part of it stays the same) using the fractions in SplitMaterial. In P3 this happens after recycling, downcycling, and energy recovery are calculated. The only flows affected are all primary flows upstream of P3. For the waste calculation an auxiliary variable is saved, $F\_2\_3\_prim\_pre\_split$ and used to calculate $F\_3_\11_a/b$. `F_2_3_prim_{alloy} = split_fraction × F_2_3_prim_{steel}`

Include check if each row sums up to 1, if not warn

## Convention 

- If a flow is multiplied by a factor with fewer dimensions the missing dimensions are treated equally.
- $F\_x\_y\_sec_{m,g,t}=F\_x\_y_{e=1,m,g,t}$  and $F\_x\_y\_prim_{m,g,t}=F\_x\_y_{e=0,m,g,t}$ in the model code $F\_x\_y_{e,m,g,t}$ should be used $\_sec/prim$ are added here to increase readability
- The _v at the end of the filenames should indicate an optional version number.
- All aggregates are useable interchangeably
- t=-1 or t=0 doesn't refer to the year being actually 0 or -1. But to the SwitchYear=0, the year in which the stock driven calculation starts and SwitchYear -1 is the year before.
- There are different aggregates namend as materials, for example aggregates, aggr_4concr and aggr_4asph. These differentiation is made such that one can distinguish where the demand for aggregates originates. These materials can be used interchangeable, in MISO2_stock_driven_properties.xlsx, SB_Material there is a column called aggregates. If a material is marked TRUE there it is an aggregate, can be used interchangable, e.g. recycled into one another, but should be accounted separately.  

## Sequence

Each calculation timestep should follow the following sequence: 

- With the function projections and the existing stock the function gross addition to stocks and the function end of life flows, $FF\_GAS$ and $FF\_EOL$.
- Using the material intensity the function stock, the gross additon as well as the EOL flows are translated into material terms.
- The total material flows from $F\_9\_10\_total$ all through $required\_total\_production$
- Flows of recycled, downcycled and energy recovery from $S\_13_{m,g,t,c\leq t-1}$ waste stock, and expiry $F\_13\_14$
- The flow of recycled material is compared to $required\_total\_production$ and $secondary\_share$ calculated
- Calculate all secondary flows from, method depending on the availability of secondary shares in the trade flows, from $F\_16\_4\_sec_{m,g,t}=F\_16\_4_{e=1,m,g,t}$ to $F\_9\_10\_sec_{m,g,t}=F\_9\_10_{e=1,m,g,t}$
- Calculate the primary follows including domestic extraction and waste from domestic extraction and raw product refining. 
- Calculate all waste flows, waste trade and add them to $S\_13_{c=t}$
- Calculate Flows to environment $F\_11\_14$ and $F\_14\_0$
  
**Optional**
- See if the function stock for energy production is big enough and increase the projection by the missing production capacity 